# Data Preprocessing

In [ ]:
from src.data import load_raw
from src.eda import effect_size

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_curve, auc
from sklearn.pipeline import Pipeline, FunctionTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.compose import ColumnTransformer

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# Import parquet/csv from data/raw


# data (as pandas dataframes) 
X = 
y = 

# Flatten y to 1D if it's a single-column DataFrame
y = y.iloc[:, 0]

In [ ]:
# Establish variables for later use
RANDOM_STATE = 42
NORMAL_CLASSES = {"Thing_Speak", "MQTT_Publish", "Wipro_bulb"}
TARGET_COL = "target"

# Focus on numeric features for correlation annd plotting
X_num = df.drop(columns=[TARGET_COL]).select_dtypes(include=[np.number])
NUMERIC_FEATURES = X_num.columns.tolist()

# Create masks for normal vs attack classes
NORMAL_MASK = df[TARGET_COL].isin(NORMAL_CLASSES)
ATTACK_MASK = ~NORMAL_MASK

In [ ]:
#| output: false

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X columns:", list(X.columns)[:10], "...")
print("y column:", y.name)


## Create binary label (y_bin):

For our baseline logistic regression, we will encode our multi-class attack types into binary.
- Converts multiclass Attack_type into binary: 0 = normal, 1 = attack
- Normal classes are the benign IoT device activity buckets: Thing_Speak, MQTT_Publish, and Wipro_bulb

In [ ]:
#| label: tbl-rtiot-lr-binary
#| tbl-cap: "Binary label distribution (0=normal, 1=attack)"
#| echo: false
#| tbl-pos: H

# 1 = attack, 0 = normal
y_bin = (~y.isin(NORMAL_CLASSES)).astype(int)

# Build a single clean summary table
class_counts = y_bin.value_counts().sort_index().rename_axis("class").to_frame("count")
class_counts["rate"] = class_counts["count"] / class_counts["count"].sum()

display(class_counts)


## Check and remove duplicate rows:

In [ ]:
#| label: tbl-rtiot-lr-deduplication
#| tbl-cap: "Deuplicate row summary for RT-IoT dataset"

# Simple duplicate check
dup_count = X.duplicated().sum()

# Drop duplicates and report how many were removed
df_nodup = df.drop_duplicates()

dup_summary = pd.DataFrame([{
    "n_rows": len(X),
    "duplicate_rows": int(dup_count),
    "duplicate_rate": dup_count / len(X),
    "Rows removed:" (len(df) - len(df_nodup))
}])

# Recover X and y from the deduplicated dataframe
X = df_nodup.drop(columns="Attack_type")
# 1 = attack, 0 = normal
y_bin = (~df_nodup["Attack_type"].isin(NORMAL_CLASSES)).astype(int)

display(dup_summary)

Rows removed: 5195


,n_rows,duplicate_rows,duplicate_rate
0,123117,5202,0.042252


## Downsampling of Attack Traffic:

The RT-IoT dataset exhibited substantial class imbalance due to the absence of a large subset of normal traffic observations (Amazon-Alexa). To prevent the classifier from learning a trivial majority-class decision rule, we constructed a balanced training dataset using random downsampling of the majority class.

Specifically, all normal traffic samples were retained, and an equal number of attack observations were randomly selected without replacement from the larger attack class. This procedure produced a 1:1 class ratio while preserving only authentic network flows, avoiding the introduction of synthetic data that could distort traffic distributions.

Downsampling was performed prior to the train–test split to ensure that duplicated or highly similar attack flows were not unevenly distributed across evaluation partitions. The resulting balanced dataset enables fair comparison of model performance by reducing bias toward the majority class and encouraging the model to learn discriminative patterns between normal and malicious network behavior.

In [ ]:
#| label: tbl-rtiot-lr-balanced
#| tbl-cap: "Balanced dataset distribution (after downsampling attacks)"

# Separate indices
idx_normal = y_bin[y_bin == 0].index
idx_attack = y_bin[y_bin == 1].index

n_normal = len(idx_normal)

# Downsample attacks
idx_attack_down = np.random.RandomState(RANDOM_STATE).choice(
    idx_attack,
    size=n_normal,
    replace=False
)

# Combine + shuffle
idx_balanced = idx_normal.append(pd.Index(idx_attack_down))
idx_balanced = (
    pd.Series(idx_balanced)
    .sample(frac=1.0, random_state=RANDOM_STATE)
    .values
)

# Balanced dataset
X_bal = X.loc[idx_balanced]
y_bal = y_bin.loc[idx_balanced]

# --- Build a clean summary table for the balanced labels ---
bal_counts = y_bal.value_counts().sort_index().rename_axis("class").to_frame("count")
bal_counts["rate"] = bal_counts["count"] / bal_counts["count"].sum()

display(bal_counts)

Balanced label distribution after downsampling (0=normal, 1=attack):


,count,rate
class,,
0,12015,0.5
1,12015,0.5


## Split train & test (80/20)

In [ ]:
#| label: tbl-rtiot-lr-split
#| tbl-cap: "Train/test split distribution for target (balanced dataset)"

X_train, X_test, y_train, y_test = train_test_split(
    X_bal,
    y_bal,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_bal,
)
print("Target train distribution:", y_train.value_counts().to_dict())
print("Target test distribution:", y_test.value_counts().to_dict())

Target train distribution: {1: 9612, 0: 9612}
Target test distribution: {0: 2403, 1: 2403}
